In [ ]:
from sentence_transformers import (
  CrossEncoder,
  InputExample,
  losses,
  evaluation,
  SentenceTransformer,
  SimilarityFunction,
  SentenceTransformerTrainer,
  SentenceTransformerTrainingArguments,
  )
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
import math
from datasets import load_dataset, Dataset, DatasetDict, Features, Value
import json
import torch
from torch.utils.data import DataLoader
from torch.quantization import quantize_dynamic
from datetime import datetime
import random
from transformers import AutoModelForSequenceClassification
import math
import pandas as pd
import os

## CrossEncoder Model - ms-marco-MiniLM-L-6-v2

In [ ]:
data_ratio = 1

# organize the data
with open('./train.json', 'r') as f:
    train = json.load(f)
random.shuffle(train)

with open('./cv.json', 'r') as f:
    cv = json.load(f)
random.shuffle(cv)

with open('./test.json', 'r') as f:
    test = json.load(f)
random.shuffle(test)

train_data = [InputExample(texts=x, label=y) for [x, y] in train[0:math.floor(len(train) * data_ratio)]]
cv_data = [InputExample(texts=x, label=y) for [x, y] in cv[0:math.floor(len(cv) * data_ratio)]]
test_data = [InputExample(texts=x, label=y) for [x, y] in test[0:math.floor(len(test) * data_ratio)]]

print(train_data[0].texts)

# training configs
train_batch_size = 256
eval_batch_size = 64
num_epochs = 10
warmup_steps = math.ceil(len(train) * num_epochs * 0.1)

In [ ]:
# initialize the model
CE_model = CrossEncoder(
  "cross-encoder/ms-marco-MiniLM-L-6-v2",
  device="mps" if torch.backends.mps.is_available() else "cpu",
  default_activation_function=torch.nn.Sigmoid()
)

CE_model_save_path = "output/training_CE_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# train the model, or load it from HuggingFace in the next cell
CE_model.fit(
    train_dataloader=DataLoader(
        dataset=train_data,
        shuffle=True,
        batch_size=train_batch_size,
        pin_memory=True),
    evaluator=CEBinaryClassificationEvaluator.from_input_examples(cv_data),
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path=CE_model_save_path
)

# write the configs
with open(CE_model_save_path + "/config.txt", 'w') as f:
    f.write("train batch size: " + str(train_batch_size))
    f.write("num epochs: " + str(num_epochs) + "\n")
    f.write("warmup_steps: " + str(warmup_steps) + "\n")
    f.write("data_ratio: " + str(1 / data_ratio) + "\n")

In [ ]:
# ... or load already trained model from local files/ HuggingFace
CE_model = CrossEncoder('nphach/jp-parallel-gloss', default_activation_function=torch.nn.Sigmoid(), device="mps" if torch.backends.mps.is_available() else "cpu")
torch.save(CE_model.model.state_dict(), "original_model.pth")
original_size = os.path.getsize("original_model.pth") / (1024 * 1024)

In [ ]:
# quantized model
Q_model = CrossEncoder('nphach/jp-parallel-gloss', default_activation_function=torch.nn.Sigmoid())
Q_model.model = quantize_dynamic(
    CE_model.model,
    {torch.nn.Linear},
    dtype=torch.qint8
)
torch.save(Q_model.model.state_dict(), "quantized.pth")
quantized_size = os.path.getsize("quantized.pth") / (1024 * 1024)

print(f"original size: {original_size:.2f} mb")
print(f"quantized size: {quantized_size:.2f} mb")
print(f"reduction: {original_size - quantized_size:.2f} mb ({(1 - quantized_size/original_size)*100:.1f}%)")

print(CE_model.rank('glass',['cup', 'sleeve']))
print(Q_model.rank('glass',['cup', 'sleeve']))


Though I was able to acheive good results with the CrossEncoder model, I'm not able to use this in production for Kotoba Tag. There's not enough resources available on my free-tier Render web service to support the model, even after quantization (lol). Offloading the model isn't an option because CrossEncoders are incompatible with all third-party inference APIs (like Transformers.js and HuggingFace Inference). We'll utilize a standard SentenceTransformer model 'all-MiniLM-L6-v2' that is relatively small and for general-use purposes.

## SentenceTransformer Model - all-MiniLM-L6-v2

In [1]:
from sentence_transformers import losses, evaluation, SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import BinaryClassificationEvaluator
import random
import json
import math
import torch
import pandas as pd
from datasets import Dataset, DatasetDict
from datetime import datetime


In [ ]:
# organize the data
data_ratio = 1 # to reduce dataset size

with open('../data/model/train.json', 'r') as f:
    train_json = json.load(f)
random.shuffle(train_json)

with open('../data/model/cv.json', 'r') as f:
    cv_json = json.load(f)
random.shuffle(cv_json)

with open('../data/model/test.json', 'r') as f:
    test_json = json.load(f)
random.shuffle(test_json)

# convert to pandas
train_df = pd.DataFrame({
    'text1': [x[0][0] for x in train_json[0:math.floor(len(train_json) * data_ratio)]],
    'text2': [x[0][1] for x in train_json[0:math.floor(len(train_json) * data_ratio)]],
    'label': [x[1] for x in train_json[0:math.floor(len(train_json) * data_ratio)]],
})
cv_df = pd.DataFrame({
    'text1': [x[0][0] for x in cv_json[0:math.floor(len(cv_json) * data_ratio)]],
    'text2': [x[0][1] for x in cv_json[0:math.floor(len(cv_json) * data_ratio)]],
    'label': [x[1] for x in cv_json[0:math.floor(len(cv_json) * data_ratio)]],
})
test_df = pd.DataFrame({
    'text1': [x[0][0] for x in test_json[0:math.floor(len(test_json) * data_ratio)]],
    'text2': [x[0][1] for x in test_json[0:math.floor(len(test_json) * data_ratio)]],
    'label': [x[1] for x in test_json[0:math.floor(len(test_json) * data_ratio)]],
})

# now to DatasetDict
data = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'cv': Dataset.from_pandas(cv_df),
    'test': Dataset.from_pandas(test_df)
})

print(f'{len(train_df) + len(cv_df) + len(test_df)} total examples')

In [ ]:
# load the all-MiniLM-L6-v2 model to finetune
ST_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="mps" if torch.backends.mps.is_available() else "cpu"
)

# define a loss function, CoSENTLoss expects a 'label' column and two non-'label' columns
loss = losses.CoSENTLoss(ST_model)

# create an evaluator
evaluator = evaluation.BinaryClassificationEvaluator(
    sentences1=data['cv']['text1'],
    sentences2=data['cv']['text2'],
    labels=data['cv']['label'],
    name='dev'
)

# specify training args
args = SentenceTransformerTrainingArguments(
  output_dir="output/ST_" + datetime.now().strftime("%m-%d_%H-%M-%S"),
  num_train_epochs=8,
  per_device_train_batch_size=128,
  per_device_eval_batch_size=32,
  warmup_ratio=0.1,
  weight_decay=0.01,
  eval_strategy="steps",
  eval_steps=5000,
  save_strategy="steps",
  save_steps=5000
)

# create a trainer and train
trainer = SentenceTransformerTrainer(
    model=ST_model,
    args=args,
    train_dataset=data['train'],
    eval_dataset=data['cv'],
    loss=loss,
    evaluator=evaluator
)
trainer.train()

In [11]:
# compare base and model results
base_model = SentenceTransformer("all-MiniLM-L6-v2", device="mps" if torch.backends.mps.is_available() else "cpu")
trained_model = SentenceTransformer("./final", device="mps" if torch.backends.mps.is_available() else "cpu")

In [12]:
print("base model eval:")
base_eval = evaluator(base_model)
print(base_eval)

print("cross validation eval:")
trained_eval = evaluator(trained_model)
print(trained_eval)

base model eval:
{'dev_cosine_accuracy': 0.9264059469941823, 'dev_cosine_accuracy_threshold': 0.3869861662387848, 'dev_cosine_f1': 0.7616255560048524, 'dev_cosine_f1_threshold': 0.34315553307533264, 'dev_cosine_precision': 0.7784666253358132, 'dev_cosine_recall': 0.7454977241242826, 'dev_cosine_ap': 0.8437392225970991, 'dev_cosine_mcc': 0.7165317489526577}
cross validation eval:
{'dev_cosine_accuracy': 0.9901422107304461, 'dev_cosine_accuracy_threshold': 0.4256893992424011, 'dev_cosine_f1': 0.9698407989716207, 'dev_cosine_f1_threshold': 0.42295974493026733, 'dev_cosine_precision': 0.9691699604743083, 'dev_cosine_recall': 0.9705125667920047, 'dev_cosine_ap': 0.993738705814257, 'dev_cosine_mcc': 0.9639493479596528}


In [ ]:
embed1 = base_model.encode(data['cv']['text1'])
embed2 = base_model.encode(data['cv']['text2'])

base_pred = base_model.similarity(embed1, embed2).diagonal()
res = pd.DataFrame({
  'text1': data['cv']['text1'],
  'text2': data['cv']['text2'],
  'pred': base_pred.numpy(),
  'truth': data['cv']['label']
})
res.to_csv('base_results.csv', index=False)

embed1 = trained_model.encode(data['cv']['text1'])
embed2 = trained_model.encode(data['cv']['text2'])

trained_pred = trained_model.similarity(embed1, embed2).diagonal()
res = pd.DataFrame({
  'text1': data['cv']['text1'],
  'text2': data['cv']['text2'],
  'pred': trained_pred.numpy(),
  'truth': data['cv']['label']
})
res.to_csv('final_results.csv', index=False)

In [17]:
res = pd.DataFrame({
  'text1': data['cv']['text1'],
  'text2': data['cv']['text2'],
  'base pred': base_pred,
  'trained pred': trained_pred,
  'base label': [True if (x >= base_eval['dev_cosine_accuracy_threshold']) else False for x in base_pred],
  'trained label': [True if (x >= trained_eval['dev_cosine_accuracy_threshold']) else False for x in trained_pred],
  'truth': data['cv']['label']
})
res.to_csv('results_comparison.csv', index=False)

In [14]:
test = evaluation.BinaryClassificationEvaluator(
    sentences1=data['test']['text1'],
    sentences2=data['test']['text2'],
    labels=data['test']['label'],
    name='test'
)
print(test(trained_model))

{'test_cosine_accuracy': 0.9897545950802664, 'test_cosine_accuracy_threshold': 0.4331962466239929, 'test_cosine_f1': 0.9685565783209015, 'test_cosine_f1_threshold': 0.4324696958065033, 'test_cosine_precision': 0.9696722939424032, 'test_cosine_recall': 0.9674434272579558, 'test_cosine_ap': 0.9934008701351884, 'test_cosine_mcc': 0.9624377824608901}
